# MINST Digit Recognizer: Comparing KNN, MLP, and Keras Model Performance
- [Rockhurst University](https://www.rockhurst.edu/)
- Predictive Modeling (BIA 6303) - Final Project
- [Chris Tan](https://www.linkedin.com/in/christan/)
- 5/12/2019

MNIST ("Modified National Institute of Standards and Technology") is the de facto “hello world” dataset of computer vision. Since its release in 1999, this classic [dataset of handwritten images](https://en.wikipedia.org/wiki/MNIST_database) has served as the basis for benchmarking classification algorithms. As new machine learning techniques emerge, MNIST remains a reliable resource for researchers and learners alike.

In this [Kaggle competition](https://www.kaggle.com/c/digit-recognizer/overview), my goal is to correctly identify digits from a dataset of tens of thousands of handwritten images. This notebook allows me to experiment with different algorithms to learn first-hand what works well and how techniques compare.

# Exploratory Data Analysis

In this section, I will focus on basic data preparation steps like loading the dataset, imputing missing values, treating categorical variables, normalizing data and creating a validation set. I will follow the same steps for the KNN, MLP, and Keras models.

![EDA](https://cdn-images-1.medium.com/max/1600/1*jXfdWyIAF-nymLFnQEkHNA.png)

## Import Libraries

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

%matplotlib inline
import numpy as np # linear algebra
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
import os as os
import sys
import warnings
import time
warnings.filterwarnings('ignore')

sns.set(style='white', context='notebook', palette='deep')
np.random.seed(2)
from IPython.display import Image

# From Matplotlib
from matplotlib.colors import ListedColormap

# From Scikit Learn
from sklearn import preprocessing, decomposition, tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from astropy.table import Table, Column
from sklearn.preprocessing import LabelEncoder
import itertools

# Set DEBUG = True to produce debug results
DEBUG = False

## Check Python Version

In [ ]:
print("The Python version is %s.%s.%s." % sys.version_info[:3])

## Check Present Working Directory

In [ ]:
%pwd

## Import Dataset

Using the [pandas.read_csv](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html) method, I read the Digit Recognizer Train and Test datasets. The data files train.csv and test.csv contain gray-scale images of hand-drawn digits, from zero through nine. Each image is 28 pixels in height and 28 pixels in width, for a total of 784 pixels in total. Each pixel has a single pixel-value associated with it, indicating the lightness or darkness of that pixel, with higher numbers meaning darker. This pixel-value is an integer between 0 and 255, inclusive.

In [ ]:
# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list the files in the input directory
%time digit_train = pd.read_csv("../input/train.csv", header=0, sep=",")
%time digit_test = pd.read_csv("../input/test.csv", header=0, sep=",")

Using the [pandas.DataFrame.describe](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.describe.html) method, I generated descriptive statistics that summarize the central tendency, dispersion and shape of a dataset’s distribution, excluding NaN values. This helps me analyzes both numeric and object series, as well as DataFrame column sets of mixed data types.

In [ ]:
digit_train.describe()

The training data set, (train.csv), has 785 columns. The first column, called "label", is the digit that was drawn by the user. The rest of the columns contain the pixel-values of the associated image. Each pixel column in the training set has a name like pixelx, where x is an integer between 0 and 783, inclusive. To locate this pixel on the image, suppose that we have decomposed x as x = i * 28 + j, where i and j are integers between 0 and 27, inclusive. Then pixelx is located on row i and column j of a 28 x 28 matrix, (indexing by zero).

In [ ]:
digit_test.describe()

The test data set, (test.csv), is the same as the training set, except that it does not contain the "label" column.

Using the [pandas.DataFrame.dropna](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.dropna.html) method with the **how = 'all'** parameter, I removed all observations where all features were **NaN**.

In [ ]:
# Source: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.dropna.html
digit_train = digit_train.dropna(axis = 0, how = 'all')
digit_test = digit_test.dropna(axis = 0, how = 'all')
if DEBUG:
    #Dimensions of dataset
    print("Shape of Data", digit_train.shape)
    print("Shape of Data", digit_test.shape)
    #Colum names
    print("Colums Names", digit_train.columns)
    print("Colums Names", digit_test.columns)
    #See bottol few rows of dataset
    print(digit_train.tail())

## Target Feature Designation

Using the code below, I set the **label** feature as the target feature and moved this to the beginning of my DataFrame.

In [ ]:
# designate target variable name
targetName = 'label'
targetSeries = digit_train[targetName]
#remove target from current location and insert in collum 0
del digit_train[targetName]
digit_train.insert(0, targetName, targetSeries)
#reprint dataframe and see target is in position 0
digit_train.head()

Using [pandas.DataFrame.info](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.info.html), I printed a concise summary of the **digit_train** DataFrame. This method prints information about a DataFrame including the index dtype and column dtypes, non-null values and memory usage.

In [ ]:
digit_train.info()

From the information above, we can see that we have **int64** data types in our dataframe.

Below is a count plot of all the instances of digits 0-9 in the **digit_train** dataset.

In [ ]:
sns.countplot(digit_train['label'])

## Target Feature Conversion to Categorical

In [ ]:
print(digit_train['label'].describe())

From the analysis above, we can see that the target feature **label** has a data type of **float64**. Since we are predicting a class versus a numerical value, I converted this feature to a string.

In [ ]:
if DEBUG:
    print(digit_train.dtypes)

In [ ]:
digit_train['label'] = digit_train['label'].astype(str)
if DEBUG:
    print(digit_train['label'].describe())

## Missing Value Replacement (NaN)

The **digit_train** dataset does not contain any NaN values. That being said, I can use the [pandas.DataFrame.fillna](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.fillna.html) method to fill any NA/NaN values using the specified method (median). 

In [ ]:
#digit_train.fillna(digit_train.median(), inplace=True)
#print(digit_train.describe())

In [ ]:
#if DEBUG:
#    print(digit_train.shape)
#    print(digit_train.info())
#    print(digit_train.head())

## Train Test Split

Since the train and test data sets are provided, I used the code below to map values from the datasets.

In [ ]:
features_train = digit_train.iloc[:,1:]
target_train = digit_train.iloc[:,0]
features_test = digit_test.iloc[:,0:]

## Grayscale Normalization

The pixel values for each image are gray scaled between 0 and 255. KNN and Multi-layer Perceptron models are sensitive to feature scaling. For my analysis, I normalized these values from 0-255 to 0-1.

In [ ]:
# pixel values are gray scale between 0 and 255
# normalize inputs from 0-255 to 0-1
features_train = features_train/255.0
features_test = features_test/255.0

In [ ]:
if DEBUG:
    print(features_train)

# Model Instantiation, Fitting, and Analysis

## K-Nearest Neighbors Model

Using the [sklearn.neighbors.KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) library, I created a K-Nearest Neighbors model. The code below interated over a k-range of 1 to 5. The KNN algorithm classifies unknown data points by comparing the unknown data point to each data point in the training set. This comparison is done using a distance function or similarity metric. Then, from the k most similar examples in the training set, we accumulate the number of “votes” for each label. The category with the highest number of votes “wins” and is chosen as the overall classification.

To start my analysis, I first ran the code below to determine the best value for k. I set **n_jobs=-1** to use all processors for the parallel jobs to run for neighbors search. I also used the [ball tree](https://scikit-learn.org/stable/modules/neighbors.html#ball-tree) algorithm. To address the inefficiencies of KD Trees in higher dimensions, the ball tree data structure was developed. Where KD trees partition data along Cartesian axes, ball trees partition data in a series of nesting hyper-spheres. This makes tree construction more costly than that of the KD tree, but results in a data structure which can be very efficient on highly structured data, even in very high dimensions.

This algorithm, by default, has **leaf_size=40**. This parameter is the number of points at which to switch to brute-force. Changing leaf_size will not affect the results of a query, but can significantly impact the speed of a query and the memory required to store the constructed tree. The amount of memory needed to store the tree scales as approximately n_samples / leaf_size.

In [ ]:
start_time = time.perf_counter()
train_results = []
test_results = []
# search for an optimal value of k for KNN MOdel
k_range = list(range(1,5))
k_scores = []
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k, n_jobs=-1, algorithm='ball_tree', leaf_size=40, weights='uniform')
    scores = cross_val_score(knn, features_train, target_train, cv=10, scoring='accuracy', n_jobs=-1)
    k_scores.append(scores.mean())
if DEBUG:
    print(k_scores) 
print(time.perf_counter() - start_time, "seconds")

As you can see, this code took a very long time to run, primarily due to the size of the training dataset. The scikit-learn library is not optimized for large datasets and only utilizes the CPU for computational work.

In [ ]:
if DEBUG:
    scores = pd.DataFrame(k_scores)
    print(scores)

In [ ]:
# plot the value of K (x-axis) versus the cross-validated accuracy (y-axis)
plt.plot(k_range, k_scores)
plt.xlabel('K Value for KNN')
plt.ylabel('Cross-Validated Accuracy')
plt.title('KNN Model for Accuracy')

In [ ]:
# changing to misclassification error
MSE = [1 - x for x in k_scores]

# determining best k
optimal_k = k_range[MSE.index(min(MSE))]

# plot misclassification error vs k
plt.plot(k_range, MSE)
plt.xlabel('Number of Neighbors K')
plt.ylabel('Misclassification Error')
plt.title('KNN Model for Misclassification Error')
plt.show()

In [ ]:
print("The optimal number of neighbors is %d." % optimal_k)

In [ ]:
start_time = time.perf_counter()
#KNN train model. Call up my model and name it clf_knn
clf_knn = KNeighborsClassifier(n_neighbors=3, n_jobs=-1, algorithm='ball_tree', leaf_size=40, weights='uniform')
#Call up the model to see the parameters you can tune (and their default setting)
print(clf_knn)
#Fit clf to the training data
clf_knn = clf_knn.fit(features_train, target_train)
#Predict clf_knn model again test data
target_predicted_knn = clf_knn.predict(features_test)
print(time.perf_counter() - start_time, "seconds")

### KNN Model Cross Validation

[Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html), sometimes called out-of-sample testing, is any of various similar model validation techniques for assessing how the results of a statistical analysis will generalize to an independent data set. Cross-validation can be used to compare the performances of different predictive modeling procedures.

Learning the parameters of a prediction function and testing it on the same data is a methodological mistake: a model that would just repeat the labels of the samples that it has just seen would have a perfect score but would fail to predict anything useful on yet-unseen data. This situation is called **overfitting**. To avoid it, it is common practice when performing a (supervised) machine learning experiment to hold out part of the available data as a **test set**: X_test, y_test. The simplest way to use cross-validation is to call the [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html#sklearn.model_selection.cross_val_score) helper function on the estimator and the dataset.

In [ ]:
start_time = time.perf_counter()
#verify KNN with Cross Validation
scores_knn = cross_val_score(clf_knn, features_train, target_train, cv=10, scoring='accuracy', n_jobs=-1)
print("Cross Validation Score for each K",scores_knn)
print("Accuracy: %0.2f (+/- %0.2f)" % (scores_knn.mean(), scores_knn.std() * 2))
print(time.perf_counter() - start_time, "seconds")

The mean score and the 95% confidence interval of the score estimate for the KNN Model is 97% with a variance of 1%.

### KNN Model Kaggle Submission

In [ ]:
digit_test['Label'] = pd.Series(target_predicted_knn)
digit_test['ImageId'] = pd.Series(range(1,28001))

In [ ]:
digit_test.to_csv('submission_knn.csv', columns=["ImageId","Label"], index=False)

### KNN Model Evaluation

My submission of my KNN Model produced a 96.857% accuracy score. This is a good baseline to compare against other models.

While simple and intuitive, and though it can even obtain very good accuracy in certain situations, the KNN algorithm has a number of drawbacks. The first is that it doesn’t actually “learn” anything — if the algorithm makes a mistake, it has no way to “correct” and “improve” itself for later classifications. Secondly, without specialized data structures, the KNN algorithm scales linearly with the number of data points, making it a questionable choice for large datasets.

## Multi-layer Perceptron Classifier

Using the [sklearn.neural_network.MLPClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html#sklearn.neural_network.MLPClassifier) library, I created three Multi-layer Perceptron Classification Model. This model optimizes the log-loss function using LBFGS or stochastic gradient descent. MLP is a class of [feedforward](https://en.wikipedia.org/wiki/Feedforward_neural_network) [artificial neural network](https://en.wikipedia.org/wiki/Artificial_neural_network) that consists of at least three layers of nodes: an input layer, a hidden layer and an output layer. Except for the input nodes, each node is a neuron that uses a nonlinear activation function.

[MLP](https://scikit-learn.org/stable/modules/neural_networks_supervised.html) is a supervised learning algorithm that learns a function by training on a dataset, where m is the number of dimensions for input and o is the number of dimensions for output. Given a set of features X = x1,x1,...,xm and a target y, it can learn a non-linear function approximator for either classification or regression. It is different from logistic regression, in that between the input and the output layer, there can be one or more non-linear layers, called hidden layers. Figure 1 shows a one hidden layer MLP with scalar output.

![Figure 1 : One hidden layer MLP.](https://scikit-learn.org/stable/_images/multilayerperceptron_network.png)

MLP utilizes a supervised learning technique called [backpropagation](https://en.wikipedia.org/wiki/Backpropagation) for training. More precisely, it trains using some form of gradient descent and the gradients are calculated using backpropagation. Its multiple layers and non-linear activation distinguish MLP from a linear perceptron in that it can distinguish data that is not linearly separable.

The leftmost layer, known as the input layer, consists of a set of neurons {xi|x1,x2,...,xm} representing the input features. Each neuron in the hidden layer transforms the values from the previous layer with a weighted linear summation w1x1+w2x2+...+wmxm, followed by a non-linear activation function g(.) : R -> R - like the hyperbolic tan function. The output layer receives the values from the last hidden layer and transforms them into output values. I chose to set **hidden_layer_sizes=(784,)** as I have 784 input features.

In [ ]:
start_time = time.perf_counter()
from sklearn.neural_network import MLPClassifier
# Multi-layer Perceptron train model. Call up my model and name it clf_mlp
clf_mlp = MLPClassifier(hidden_layer_sizes=(784,), warm_start=True)
#Call up the model to see the parameters you can tune (and their default setting)
print(clf_mlp)
#Fit clf_NN to the training data
clf_mlp = clf_mlp.fit(features_train, target_train)
#Predict clf_NN model again test data
target_predicted_mlp = clf_mlp.predict(features_test)
print(time.perf_counter() - start_time, "seconds")

Building this model was slightly faster than the KNN model.

### Multi-layer Perceptron Classifier Model Cross Validation

In [ ]:
start_time = time.perf_counter()
#verify RF with Cross Validation
scores_mlp = cross_val_score(clf_mlp, features_train, target_train, cv=10, n_jobs=-1, scoring='accuracy')
print("Cross Validation Score for MLP",scores_mlp)
print("Accuracy: %0.2f (+/- %0.2f)" % (scores_mlp.mean(), scores_mlp.std() * 2))
print(time.perf_counter() - start_time, "seconds")

The mean score and the 95% confidence interval of the score estimate for the Multi-layer Perceptron Classifier Model is 98% with a variance of 1%. This is an improvement on the KNN Model with similar variance.

### Multi-layer Perceptron Classifier Model Kaggle Submission

In [ ]:
digit_test['Label'] = pd.Series(target_predicted_mlp)
digit_test['ImageId'] = pd.Series(range(1,28001))

In [ ]:
digit_test.to_csv('submission_mlp.csv', columns=["ImageId","Label"], index=False)

### Multi-layer Perceptron Model Evaluation

My submission of my Multi-layer Perceptron Model produced an accuracy score of 98.000%, much better than my KNN Model.

The advantages of Multi-layer Perceptron are:
- Capability to learn non-linear models.
- Capability to learn models in real-time (on-line learning) using partial_fit.

The disadvantages of Multi-layer Perceptron (MLP) include:
- MLP with hidden layers have a non-convex loss function where there exists more than one local minimum. Therefore different random weight initializations can lead to different validation accuracy.
- MLP requires tuning a number of hyperparameters such as the number of hidden neurons, layers, and iterations.
- MLP is sensitive to feature scaling.

Per scikit-learn documentation, neural network models in scikit are not intended for large-scale applications. In particular, scikit-learn offers no GPU support.

## Keras Model

Convolutional Neural Networks are part of deep, feed forward artificial neural networks that can perform a variety of task with even better time and accuracy than other classifiers, in different applications of image and video recognition, recommender system and natural language processing.

![CNN](https://cdn-images-1.medium.com/max/800/1*22R-AyQ-oXb8Flod9PsyNw.png)

[Keras](https://keras.io/) is a high-level neural networks API, written in Python and capable of running on top of [TensorFlow](https://www.tensorflow.org/), [CNTK](https://docs.microsoft.com/en-us/cognitive-toolkit/), or [Theano](http://deeplearning.net/software/theano/). It was developed with a focus on enabling fast experimentation. Being able to go from idea to result with the least possible delay is key to doing good research.

[Siraj Raval](https://www.youtube.com/channel/UCWN3xxRkmTPmbKwht9FuE5A) from [The School of AI](https://www.theschool.ai/) posted an excellent YouTube video explaining Keras. I've embedded this below.

In [ ]:
from IPython.lib.display import YouTubeVideo
vid = YouTubeVideo('j_pJmXJwMLA', autoplay=0)
display(vid)

Use Keras if you need a deep learning library that:
- Allows for easy and fast prototyping (through user friendliness, modularity, and extensibility).
- Supports both convolutional networks and recurrent networks, as well as combinations of the two.
- Runs seamlessly on CPU and GPU.


In [ ]:
# From Keras for TensorFlow Model
import keras as keras
from keras.utils.np_utils import to_categorical # convert to one-hot-encoding
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D
from keras.optimizers import RMSprop
from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import ReduceLROnPlateau

### Reshape

Train and test images (28px x 28px) have been stock into [pandas.Dataframe](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html) as 1D vectors of 784 values. I reshaped all data to 28x28x1 3D matrices.

In [ ]:
# Reshape image in 3 dimensions (height = 28px, width = 28px , canal = 1)
X_train = features_train.values.reshape(-1,28,28,1)
test = features_test.values.reshape(-1,28,28,1)

Keras requires an extra dimension in the end which correspond to channels. MNIST images are gray scaled so it use only one channel. For RGB images, there is 3 channels, we would have reshaped 784px vectors to 28x28x3 3D matrices.

### Label Encoding

Labels are 10 digits numbers from 0 to 9. We need to encode these lables to one hot vectors (ex : 2 -> [0,0,1,0,0,0,0,0,0,0]). This is also required as I used [categorical_crossentropy](https://www.tensorflow.org/api_docs/python/tf/keras/backend/categorical_crossentropy) as my loss function.

In [ ]:
# Encode labels to one hot vectors (ex : 2 -> [0,0,1,0,0,0,0,0,0,0])
Y_train = to_categorical(target_train, num_classes = 10)

### Train/Val Split

I split the train set in two parts : a small fraction (10%) became the validation set which the model is evaluated and the rest (90%) is used to train the model.

In [ ]:
# Split the train and the validation set for the fitting
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size = 0.1, random_state=0)

Below is an example of one of the images rendered from the pixel data.

In [ ]:
g = plt.imshow(X_train[10][:,:,0])

### Model Buidling

In this section, I will define my convolutional neural network model, compile it, and fit it against the digits datasets.

![Model Building](https://cdn-images-1.medium.com/max/800/1*uaE9vcY1M2-RTCpurKr9lg.png)

### Core Layers

To begin building my model, I created a Sequential model. This is a linear stack of layers. The model needs to know what input shape it should expect. For this reason, the first layer in a Sequential model (and only the first, because following layers can do automatic shape inference) needs to receive information about its input shape. There are several possible ways to do this:
- Pass an input_shape argument to the first layer. This is a shape tuple (a tuple of integers or None entries, where None indicates that any positive integer may be expected). In input_shape, the batch dimension is not included.
- Some 2D layers, such as Dense, support the specification of their input shape via the argument input_dim, and some 3D temporal layers support the arguments input_dim and input_length.
- If you ever need to specify a fixed batch size for your inputs (this is useful for stateful recurrent networks), you can pass a batch_size argument to a layer. If you pass both batch_size=32 and input_shape=(6, 8) to a layer, it will then expect every batch of inputs to have the batch shape (32, 6, 8).

In [ ]:
clf_keras = Sequential()

### Convolutional Layers

For the first input layer, I used a [Conv2D](https://keras.io/layers/convolutional/#conv2d) layer, a 2D convolution layer (e.g. spatial convolution over images). This layer creates a convolution kernel that is convolved with the layer input to produce a tensor of outputs. I also added [BatchNormalization](https://keras.io/layers/normalization/) after each Conv2D layer. This normalizes the activations of the previous layer at each batch, i.e. applies a transformation that maintains the mean activation close to 0 and the activation standard deviation close to 1. Lastly, I added the [MaxPooling2D](https://keras.io/layers/pooling/) layer for max pooling operation for spatial data.

In [ ]:
clf_keras.add(Conv2D(filters = 32, kernel_size = (5,5), padding = 'same', strides=1, activation ='relu', 
                     input_shape = (28,28,1)))
clf_keras.add(Conv2D(filters = 32, kernel_size = (5,5), padding = 'same', activation ='relu'))
clf_keras.add(keras.layers.BatchNormalization())
clf_keras.add(keras.layers.MaxPooling2D(pool_size=(2,2)))

Conv2D Arguments
- **filters**: is the number of desired feature maps.
- **kernel_size**: is the size of the convolution kernel. A single number 5 means a 5x5 convolution.
- **strides**: the new layer maps will have a size equal to the previous layer maps divided by strides. Leaving this blank results in strides=1.
- **padding**: is either 'same' or 'valid'. Leaving this blank results in padding='valid'. If padding is 'valid' then the size of the new layer maps is reduced by kernel_size-1. For example, if you perform a 5x5 convolution on a 28x28 image (map) with padding='valid', then the next layer has maps of size 24x24. If padding is 'same', then the size isn't reduced.
- **activation**: is applied during forward propagation. Leaving this blank results in no activation.

MaxPooling2D Arguments
- **pool_size**: integer or tuple of 2 integers, factors by which to downscale (vertical, horizontal). (2, 2) will halve the input in both spatial dimension. If only one integer is specified, the same window length will be used for both dimensions.
- **strides**: Integer, tuple of 2 integers, or None. Strides values. If None, it will default to pool_size.
- **padding**: One of "valid" or "same" (case-insensitive).
- **data_format**: A string, one of channels_last (default) or channels_first. The ordering of the dimensions in the inputs.  channels_last corresponds to inputs with shape  (batch, height, width, channels) while channels_first corresponds to inputs with shape  (batch, channels, height, width). It defaults to the image_data_format value found in your Keras config file at ~/.keras/keras.json. If you never set it, then it will be "channels_last".

Per the Kaggle discussion [How to score 97%, 98%, 99%, and 100%](https://www.kaggle.com/c/digit-recognizer/discussion/61480) the best design is to add another convolution and pooling layer.

In [ ]:
clf_keras.add(Conv2D(filters = 64, kernel_size = (3,3), strides=2, padding = 'same', activation ='relu'))
clf_keras.add(Conv2D(filters = 64, kernel_size = (3,3), padding = 'same', activation ='relu'))
clf_keras.add(keras.layers.BatchNormalization())
clf_keras.add(keras.layers.MaxPooling2D(pool_size=(2,2)))

### Dropout

Dropout consists in randomly setting a fraction rate of input units to 0 at each update during training time. This is a [simple way to prevent neural networks from overfitting](http://www.jmlr.org/papers/volume15/srivastava14a/srivastava14a.pdf).

In [ ]:
clf_keras.add(Dropout(rate = 0.5))

Arguments
- **rate**: float between 0 and 1. Fraction of the input units to drop.
- **noise_shape**: 1D integer tensor representing the shape of the binary dropout mask that will be multiplied with the input. For instance, if your inputs have shape  (batch_size, timesteps, features) and you want the dropout mask to be the same for all timesteps, you can use noise_shape=(batch_size, 1, features).
- **seed**: A Python integer to use as random seed.

### Flatten

The parameter below [flattens](https://keras.io/layers/core/#flatten) the input. Does not affect the batch size.

In [ ]:
clf_keras.add(Flatten())

Arguments
- **data_format**: A string, one of channels_last (default) or channels_first. The ordering of the dimensions in the inputs. The purpose of this argument is to preserve weight ordering when switching a model from one data format to another.  channels_last corresponds to inputs with shape  (batch, ..., channels) while channels_first corresponds to inputs with shape (batch, channels, ...). It defaults to the image_data_format value found in your Keras config file at ~/.keras/keras.json. If you never set it, then it will be "channels_last".

### Dense Layer and Activations

[Dense](https://keras.io/layers/core/#dense) implements the operation: output = activation(dot(input, kernel) + bias) where activation is the element-wise activation function passed as the activation argument, kernel is a weights matrix created by the layer, and bias is a bias vector created by the layer (only applicable if use_bias is True).

[Activations](https://keras.io/layers/core/#activation) can either be used through an **Activation** layer, or through the **activation** argument supported by all forward layers. Below I used the Softmax activation function.

In [ ]:
clf_keras.add(Dense(10, activation = "softmax"))

Arguments
- **x**: Input tensor.
- **axis**: Integer, axis along which the softmax normalization is applied.

### Optimizers

An [optimizer](https://keras.io/optimizers/) is one of the two arguments required for compiling a Keras model. I chose the [Nesterov Adam](http://cs229.stanford.edu/proj2015/054_report.pdf) optimizer. Much like Adam is essentially RMSprop with momentum, Nadam is Adam RMSprop with Nesterov momentum. Default parameters follow those provided in the paper. It is recommended to leave the parameters of this optimizer at their default values. More information can be found in the whitepaper ["On the importance of initialization and momentum in deep learning"](http://www.cs.toronto.edu/~fritz/absps/momentum.pdf).

In [ ]:
# Define the optimizer
optimizer = keras.optimizers.Nadam(lr=0.002, beta_1=0.9, beta_2=0.999, epsilon=None, schedule_decay=0.004)

### Compilation

Before training my model, I need to configure the learning process, which is done via the compile method. It receives three arguments:
- An **optimizer**. This could be the string identifier of an existing optimizer (such as rmsprop or adagrad), or an instance of the Optimizer class. See: [optimizers](https://keras.io/optimizers/).
- A **loss** function. This is the objective that the model will try to minimize. It can be the string identifier of an existing loss function (such as categorical_crossentropy or mse), or it can be an objective function. See: [losses](https://keras.io/losses/).
- A list of **metrics**. For any classification problem you will want to set this to metrics=['accuracy']. A metric could be the string identifier of an existing metric or a custom metric function.

In [ ]:
# Compile the model
clf_keras.compile(optimizer = optimizer, loss = "categorical_crossentropy", metrics=["accuracy"])

Since my target is a 10 class categorical class, I chose the 'categorical_crossentropy' loss function.

The metric function "accuracy" is used is to evaluate the performance our model. This metric function is similar to the loss function, except that the results from the metric evaluation are not used when training the model (only for evaluation).

### Callbacks

A [callback](https://keras.io/callbacks/) is a set of functions to be applied at given stages of the training procedure. You can use callbacks to get a view on internal states and statistics of the model during training. You can pass a list of callbacks (as the keyword argument callbacks) to the .fit() method of the Sequential or  Model classes. The relevant methods of the callbacks will then be called at each stage of the training.

The [ReduceLROnPlateau](https://keras.io/callbacks/#reducelronplateau) callback reduces learning rate when a metric has stopped improving. Models often benefit from reducing the learning rate by a factor of 2-10 once learning stagnates. This callback monitors a quantity and if no improvement is seen for a 'patience' number of epochs, the learning rate is reduced.

In [ ]:
learning_rate_reduction = ReduceLROnPlateau(monitor='val_acc', factor=0.2, patience=3, min_lr=0.00001, verbose=1)

Arguments
- **monitor**: quantity to be monitored.
- **factor**: factor by which the learning rate will be reduced. new_lr = lr * factor
- **patience**: number of epochs with no improvement after which learning rate will be reduced.
- **verbose**: int. 0: quiet, 1: update messages.
- **mode**: one of {auto, min, max}. In min mode, lr will be reduced when the quantity monitored has stopped decreasing; in max mode it will be reduced when the quantity monitored has stopped increasing; in auto mode, the direction is automatically inferred from the name of the monitored quantity.
- **min_delta**: threshold for measuring the new optimum, to only focus on significant changes.
- **cooldown**: number of epochs to wait before resuming normal operation after lr has been reduced.
- **min_lr**: lower bound on the learning rate.

Our data is almost ready and we could start our training right now but there is one more thing that we could do to improve our classifier - data augmentation.

### Image Preprocessing

The [ImageDataGenerator](https://keras.io/preprocessing/image/) class generates batches of tensor image data with real-time data augmentation. The data will be looped over (in batches). The idea behind data augmentation is that we can artificially enlarge our training dataset (thus reducing overfitting) by its augmentation.

In [ ]:
datagen = ImageDataGenerator(
        featurewise_center=False,  # set input mean to 0 over the dataset
        samplewise_center=False,  # set each sample mean to 0
        featurewise_std_normalization=False,  # divide inputs by std of the dataset
        samplewise_std_normalization=False,  # divide each input by its std
        zca_whitening=False,  # apply ZCA whitening
        rotation_range=10,  # randomly rotate images in the range (degrees, 0 to 180)
        zoom_range = 0.10, # Randomly zoom image 
        width_shift_range=0.1,  # randomly shift images horizontally (fraction of total width)
        height_shift_range=0.1,  # randomly shift images vertically (fraction of total height)
        horizontal_flip=False,  # randomly flip images
        vertical_flip=False)  # randomly flip images

datagen.fit(X_train)

### Fit Generator

An epoch is an iteration over the entire data provided, as defined by steps_per_epoch. Note that in conjunction with initial_epoch, epochs is to be understood as "final epoch". The model is not trained for a number of iterations given by epochs, but merely until the epoch of index epochs is reached.

In [ ]:
epochs = 30

Batch size is the number of samples per evaluation step. 

In [ ]:
batch_size = 86

Now that our model is ready, we use fit_generator to train the model on data generated batch-by-batch by a Python generator (or an instance of Sequence). The generator is run in parallel to the model, for efficiency. For instance, this allows you to do real-time data augmentation on images on CPU in parallel to training your model on GPU. The use of keras.utils.Sequence guarantees the ordering and guarantees the single use of every input per epoch when using use_multiprocessing=True.

In [ ]:
start_time = time.perf_counter()
# Fit the model 
history = clf_keras.fit_generator(datagen.flow(X_train,Y_train, batch_size=batch_size), epochs = epochs, 
                                  validation_data = (X_val,Y_val), verbose = 2, 
                                  steps_per_epoch=features_train.shape[0] // batch_size, 
                                  callbacks=[learning_rate_reduction], 
                                  use_multiprocessing=True)
print(time.perf_counter() - start_time, "seconds")

This returns a History object. Its History.history attribute is a record of training loss values and metrics values at successive epochs, as well as validation loss values and validation metrics values (if applicable).

### Training History Visualizaton

The fit() method on a Keras Model returns a History object. The History.history attribute is a dictionary recording training loss values and metrics values at successive epochs, as well as validation loss values and validation metrics values (if applicable). 

In [ ]:
# Plot training & validation accuracy values
plt.plot(history.history['acc'])
plt.plot(history.history['val_acc'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()

# Plot training & validation loss values
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()

### Keras Model Kaggle Submission

In [ ]:
# predict results
results = clf_keras.predict(test)

# select the indix with the maximum probability
results = np.argmax(results,axis = 1)

results = pd.Series(results,name="Label")

In [ ]:
submission = pd.concat([pd.Series(range(1,28001),name = "ImageId"),results],axis = 1)

In [ ]:
submission.to_csv("submission_keras.csv",index=False)

### Keras Model Evaluation

The submission of my Keras Model produced a 99.571% accuracy score. This is a significant boost over my KNN and MLP Models.

## Summary

I noticed gradual improvements as the models progressed from K-Nearest Neighbors to Multi-layer Perceptron to Keras. While KNN and MLP produced good results and was easy to implement, the process to gain these results was computationally expensive. The Keras Model produced the best accuracy of the three models and the time to run the model was considerably less than the KNN and MLP models. That being said, there is a bit of a learning curve when implementing a Keras deep learning model.

For this type of machine learning problem, [scikit-learn](https://scikit-learn.org/stable/) is not the best Python library to use. Scikit-learn is not capable of utilizing a GPU for computational processing which leads to extremely long model training. When working with large datasets and convolutional neural networks, the [Keras](https://keras.io/) and [TensorFlow](https://www.tensorflow.org/) API libraries proved to be much better and processing the data and producing excellent results.

References
- [Digit Recognizer - Introduction to Kaggle Competitions](https://towardsdatascience.com/digit-recognizer-introduction-to-kaggle-competitions-with-image-classification-task-0-995-268fa2b90e13)
- [Introduction to CNN Keras - Acc 0.997 (top 8%)](https://www.kaggle.com/yassineghouzam/introduction-to-cnn-keras-0-997-top-6)